<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2FACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

---

# Track 1 (Notebook 1): Unified Bronze Lakehouse & Multi-Cloud Data Access in BigQuery Studio UI
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

## 🗺️ 1. Unified Bronze Lakehouse Architecture (3 Storage Engines in 1 BigQuery UI)

Before building our Silver & Gold Medallion pipelines in **Notebook 2**, this notebook establishes **BigQuery Studio as ACSM's Single Unified Data Access Layer (`RFP Clauses C1.1.1.1, C1.1.1.2, C1.1.1.3, C1.1.1.4 & C1.1.1.18`)** across **three storage engines**:
1. **BigQuery Native Storage (`6` Fact Tables in `acsm_bronze` — `1,233,284` rows)**: `Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, and `Fact_CC_Collection` ingested via serverless `LOAD DATA OVERWRITE` at **`$0` / `0 Bytes Billed`**.
2. **GCP Lakehouse Open-Source Apache Iceberg Table (`acsm_bronze.m3CIF` — `100,000` rows)**: Customer CIF Master created as a **BigLake Managed Apache Iceberg Table (`table_format = 'ICEBERG'`, `file_format = 'PARQUET'`)** stored on Google Cloud Storage (`gs://.../iceberg/m3CIF`) with `EXPORT TABLE METADATA`, **Schema Evolution (`ADD COLUMN`)**, and **Time Travel (`FOR SYSTEM_TIME AS OF`)**.
3. **Cross-Cloud AWS Glue Federated Apache Iceberg Table (`acsm_aws_federated_catalog.acsm_aws_bronze.dimProduct` — `65,000` rows)**: Credit Card Product Master stored in **Amazon S3 (`s3://...`) + AWS Glue Data Catalog (`<AWS_ACCOUNT_ID>` / `acsm-gcp-trust-role` / `acsm-federated-only-policy`)** and queried directly from BigQuery Studio without cross-cloud ETL!

![Track 1 Notebook 1 — Unified Bronze Lakehouse Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook1_serverless_ingestion_flow.png)

---

## 🔗 2. Cross-Cloud AWS Glue Catalog Federation Set Up (`acsm_aws_bronze.dimProduct`)

![Cross-Cloud AWS Glue Catalog Federation Set Up — Slide 6](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/slide6_aws_glue_flow.png)

| Storage Engine in Bronze | ACSM Table(s) | Row Count | Storage Format & Location |
| :--- | :--- | :---: | :--- |
| **1. BigQuery Native Storage** | `Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection` | `1,233,284` rows | BigQuery Capacitor Native Storage (`asia-southeast1`) |
| **2. GCP Lakehouse Apache Iceberg** | **`acsm_bronze.m3CIF`** *(Customer CIF Master)* | `100,000` rows | Open Apache Parquet + Iceberg `v*.metadata.json` (`gs://acsm-workshop-landing-${PROJECT_ID}/iceberg/m3CIF`) |
| **3. AWS Glue Federated Apache Iceberg** | **`acsm_aws_bronze.dimProduct`** *(Credit Card Product Master)* | `65,000` rows | Amazon S3 Apache Parquet + AWS Glue Catalog (`s3://.../acsm_aws_bronze/dimProduct/` in `ap-southeast-1`) |


---
## Step 0 (Prerequisite): Create VPC Network & Singapore (`asia-southeast1`) Subnetwork in Google Cloud Shell First

> **⚠️ IMPORTANT — RUN ONCE IN GOOGLE CLOUD SHELL BEFORE CONNECTING THIS NOTEBOOK**
> BigQuery Studio Notebooks (powered by Colab Enterprise) require a **VPC Network** and a **Regional Subnetwork in Singapore (`asia-southeast1`)** with **Private Google Access** enabled to provision the notebook runtime.
> 1. Click **Activate Cloud Shell (`>_`)** in the top-right corner of the Google Cloud Console.
> 2. Copy and run the `gcloud` commands below in **Cloud Shell**.
> 3. Once complete, come back to this Notebook in **BigQuery Studio**, click **Connect** (top-right), and select Network **`acsm-colab-network`** and Subnetwork **`acsm-colab-subnet-sg`** (`asia-southeast1`).

```bash
# =============================================================================
# Run these commands in Google Cloud Shell (>_) BEFORE connecting the notebook
# =============================================================================
export PROJECT_ID="<YOUR_GCP_PROJECT_ID>"   # e.g., export PROJECT_ID="${PROJECT_ID}"
export LOCATION="asia-southeast1"           # Always Singapore (asia-southeast1)
export NETWORK_NAME="acsm-colab-network"
export SUBNET_NAME="acsm-colab-subnet-sg"

gcloud config set project "${PROJECT_ID}"

# 1. Enable required APIs for BigQuery Studio Notebooks (Colab Enterprise)
gcloud services enable \
  bigquery.googleapis.com \
  aiplatform.googleapis.com \
  compute.googleapis.com \
  dataform.googleapis.com \
  --project="${PROJECT_ID}"

# 2. Create Custom VPC Network
gcloud compute networks create "${NETWORK_NAME}" \
  --project="${PROJECT_ID}" \
  --subnet-mode=custom

# 3. Create Regional Subnetwork in Singapore (asia-southeast1) with Private Google Access
gcloud compute networks subnets create "${SUBNET_NAME}" \
  --project="${PROJECT_ID}" \
  --network="${NETWORK_NAME}" \
  --region="${LOCATION}" \
  --range="10.10.0.0/24" \
  --enable-private-ip-google-access

# 4. Create Cloud Router & Cloud NAT in Singapore (allows git clone from GitHub without public IPs)
gcloud compute routers create "acsm-colab-router-sg" \
  --project="${PROJECT_ID}" \
  --network="${NETWORK_NAME}" \
  --region="${LOCATION}"

gcloud compute routers nats create "acsm-colab-nat-sg" \
  --project="${PROJECT_ID}" \
  --router="acsm-colab-router-sg" \
  --region="${LOCATION}" \
  --auto-allocate-nat-external-ips \
  --nat-all-subnet-ip-ranges
```

> **🔍 How to Verify Step 0 on GCP Console UI & Connect the Notebook Runtime**
> 1. Open **VPC network $\rightarrow$ VPC networks** in the GCP Console and verify **`acsm-colab-network`** and subnet **`acsm-colab-subnet-sg`** (`Region: asia-southeast1`, `Private Google access: On`) are listed.
> 2. Come back to **BigQuery Studio**, open this notebook, click the **Connect** dropdown (top-right) $\rightarrow$ **Connect to a runtime** (or **Create a runtime template** in `asia-southeast1`), select Network **`acsm-colab-network`** and Subnetwork **`acsm-colab-subnet-sg`**, and click **Connect**.

---
## Step 1: Configure Parameters (`PROJECT_ID` & Singapore Region) and Clone Repository

In [ ]:
# @title 1. Set Parameterized `PROJECT_ID` (Auto-Detects Active Project if Blank) & Singapore Region (`asia-southeast1`)
import os
import subprocess

PROJECT_ID = ""  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID.startswith("<"):
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
    )
LOCATION = "asia-southeast1"  # @param {type:"string"}
BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

# Load the native BigQuery SQL magic so all SQL cells run against $PROJECT_ID in $LOCATION
%load_ext google.cloud.bigquery

!gcloud config set project $PROJECT_ID
![ -d aeon-credit-gcp-workshop ] || git clone https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git
print(f"Active Project: {PROJECT_ID} | Region: {LOCATION} | Bucket: gs://{BUCKET_NAME}")


> **🔍 How to Verify Step 1 on GCP Console UI**
> 1. In the top Google Cloud Console bar, confirm your **`PROJECT_ID`** is selected in the Project Picker.
> 2. In the cell output above, confirm the repository cloned cleanly and `LOCATION=asia-southeast1` (Singapore) is active.
---
## Step 2: Create the Cloud Storage Landing Bucket in Singapore (`asia-southeast1`)

In [ ]:
# @title Step 2: Create Regional GCS Bucket in Singapore (asia-southeast1)
!gcloud storage buckets describe gs://{BUCKET_NAME} --project={PROJECT_ID} >/dev/null 2>&1 || \
  gcloud storage buckets create gs://{BUCKET_NAME} \
    --project={PROJECT_ID} \
    --location={LOCATION} \
    --uniform-bucket-level-access

!gcloud storage buckets describe gs://{BUCKET_NAME} --format="table(name,location,location_type,storage_class)"

> **🔍 How to Verify Step 2 on GCP Console UI (Cloud Storage Browser)**
> 1. Open **Cloud Storage $\rightarrow$ Buckets** in the GCP Console.
> 2. Verify bucket **`acsm-workshop-landing-${PROJECT_ID}`** shows **Location type**: `Region`, **Location**: `asia-southeast1 (Singapore)`, and **Public access**: `Not public`.
---
## Step 3: Copy Compressed Dataset Files (`.csv.gz`) from Repo to Singapore Bucket

In [ ]:
# @title Step 3: Upload the 7 GCP Compressed .csv.gz Files (6 Fact Tables + m3CIF; dimProduct lives in AWS S3)
!gcloud storage cp \
  aeon-credit-gcp-workshop/data/full_compressed/Fact_*.csv.gz \
  aeon-credit-gcp-workshop/data/full_compressed/m3CIF.csv.gz \
  gs://{BUCKET_NAME}/full_compressed/
!gcloud storage ls -l gs://{BUCKET_NAME}/full_compressed/


> **🔍 How to Verify Step 3 on GCP Console UI (Bucket Objects View)**
> 1. In **Cloud Storage $\rightarrow$ Buckets**, click **`acsm-workshop-landing-${PROJECT_ID}` $\rightarrow$ `full_compressed/`**.
> 2. Click **Refresh** and verify the 7 GCP `.csv.gz` files (`6` `Fact_*.csv.gz` tables + `m3CIF.csv.gz`) are listed in `asia-southeast1 (Singapore)` with `application/gzip` content type.
---
## Step 4: Storage Engine 1 — Run `CREATE TABLE` DDL for the 6 BigQuery Native Fact Tables (185 Column Descriptions)

> **📌 Architectural Note (3 Bronze Storage Engines across 8 Total Tables / 226 Columns)**:
> - **Storage Engine 1 (Step 4 & Step 5 — BigQuery Native Storage)**: Creates and loads the **6 Transactional Fact Tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection` — **`185` column descriptions**, **`1,233,284` rows**).
> - **Storage Engine 2 (Step 7 — GCP Lakehouse Open-Source Apache Iceberg)**: Creates **`acsm_bronze.m3CIF`** (**`28` column descriptions**, **`100,000` rows**) as a BigLake Managed Apache Iceberg table on GCS.
> - **Storage Engine 3 (Step 8 — Cross-Cloud AWS Glue Federated Apache Iceberg)**: Federates **`acsm_aws_bronze.dimProduct`** (**`13` columns**, **`65,000` rows**) directly from Amazon S3 + AWS Glue (`ap-southeast-1`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- =============================================================================
-- DEMO FLOW 2 (STEP 1 OF 2): Create 6 ACSM BigQuery Native Fact Tables with Table & Column Descriptions
-- Project: `<PROJECT_ID>` (e.g. `${PROJECT_ID}`) | Dataset: `acsm_bronze`
-- Location: `asia-southeast1` (Singapore)
-- Source Dictionary: `Mock Metadata.xlsx` (6 BigQuery Native Fact tables, 185 described columns; m3CIF is created as Iceberg in Step 7, dimProduct is federated from AWS Glue in Step 8)
-- =============================================================================

CREATE SCHEMA IF NOT EXISTS `acsm_bronze`
OPTIONS (
  location = "asia-southeast1",
  description = "AEON Credit Service Malaysia (ACSM) — 8 Core Tables (T1-T8) with Governed Metadata (Singapore Region)"
);

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Judge` (
  `Rcd_DT` DATE OPTIONS(description = "Data extraction date [Source Data Type: date]"),
  `CIF_NO` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: numeric]"),
  `APPL_NO` INT64 OPTIONS(description = "PRODUCT application number [Source Data Type: varchar]"),
  `AGREE_NO` INT64 OPTIONS(description = "PRODUCT loan account number [Source Data Type: numeric]"),
  `APPL_DT` INT64 OPTIONS(description = "Easy payment application date [Source Data Type: numeric]"),
  `APPL_STS` STRING OPTIONS(description = "Easy payment application status [Source Data Type: varchar]"),
  `JUDGE_DT` INT64 OPTIONS(description = "Easy payment application decision date [Source Data Type: numeric]"),
  `SCORING_POINT` INT64 OPTIONS(description = "Easy payment Final Score for decision [Source Data Type: numeric]"),
  `SCORING_TYPE` INT64 OPTIONS(description = "Easy payment Final Score type [Source Data Type: numeric]"),
  `SCORING_RANK` STRING OPTIONS(description = "Easy payment Final score rank based on bucket [Source Data Type: varchar]"),
  `LOAN_CODE` INT64 OPTIONS(description = "Loan type [Source Data Type: varchar]"),
  `LOAN_TYP_ID` INT64 OPTIONS(description = "Loan subtype [Source Data Type: varchar]"),
  `LOAN_GRP` INT64 OPTIONS(description = "Loan group [Source Data Type: varchar]"),
  `AGENT_CODE1` INT64 OPTIONS(description = "Merchant group [Source Data Type: varchar]"),
  `AGENT_CODE2` INT64 OPTIONS(description = "Merchant subgroup [Source Data Type: varchar]"),
  `APPL_CHANNEL` STRING OPTIONS(description = "Application channel [Source Data Type: varchar]"),
  `REJECT_CODE` STRING OPTIONS(description = "Reject Code [Source Data Type: varchar]"),
  `FIN_AMT` FLOAT64 OPTIONS(description = "Easy payment finance approved amount [Source Data Type: numeric]"),
  `FIN_PRFT_AMT` FLOAT64 OPTIONS(description = "Easy payment profit/interest amount [Source Data Type: numeric]"),
  `FIN_TOTAL_AMT` FLOAT64 OPTIONS(description = "Easy payment financing total amount [Source Data Type: numeric]"),
  `INST_AMT` FLOAT64 OPTIONS(description = "Easy payment installment amount [Source Data Type: numeric]"),
  `DEPOSIT` INT64 OPTIONS(description = "% for downpayment [Source Data Type: numeric]"),
  `INTEREST` FLOAT64 OPTIONS(description = "Easy payment interest [Source Data Type: decimal]"),
  `TOTAL_INST` INT64 OPTIONS(description = "Easy payment total installment months; loan tenure [Source Data Type: numeric]"),
  `JointIncome_FG` STRING OPTIONS(description = "Joint income yes/no flag [Source Data Type: varchar]"),
  `SCORE_DECISION` STRING OPTIONS(description = "Easy payment score decision: accept/decline [Source Data Type: char]"),
  `NetIncome` FLOAT64 OPTIONS(description = "Easy payment applicant net income [Source Data Type: numeric]"),
  `Income` FLOAT64 OPTIONS(description = "Easy payment applicant income [Source Data Type: numeric]"),
  `Age` INT64 OPTIONS(description = "Easy payment applicant age [Source Data Type: int]"),
  `HomeYear` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: numeric]"),
  `YearOfBusiness` INT64 OPTIONS(description = "Number of years of employment in current company [Source Data Type: numeric]"),
  `AnnualIncome` FLOAT64 OPTIONS(description = "Annual income [Source Data Type: numeric]"),
  `TOTAL_AEON_INST` INT64 OPTIONS(description = "Total installment for all Aeon products [Source Data Type: numeric]"),
  `TOTAL_AEON_OSB` FLOAT64 OPTIONS(description = "Total outstanding balance for all Aeon products [Source Data Type: numeric]"),
  `CUR_REPAY_RATIO` FLOAT64 OPTIONS(description = "Current repayment ratio [Source Data Type: decimal]"),
  `NEW_REPAY_RATIO` FLOAT64 OPTIONS(description = "New repayment ratio [Source Data Type: decimal]"),
  `NDI` FLOAT64 OPTIONS(description = "Net disposable income [Source Data Type: numeric]"),
  `CUR_DSR` FLOAT64 OPTIONS(description = "Current debt to service ratio [Source Data Type: decimal]"),
  `NEW_DSR` FLOAT64 OPTIONS(description = "New debt to service ratio [Source Data Type: decimal]"),
  `B_OtherIncome` FLOAT64 OPTIONS(description = "Other income [Source Data Type: numeric]"),
  `B_NonBankCommitment` FLOAT64 OPTIONS(description = "Non-bank commitment [Source Data Type: numeric]"),
  `DEPENDANT` INT64 OPTIONS(description = "Children, spouse. [Source Data Type: numeric]"),
  `YEAR_MADE` INT64 OPTIONS(description = "Easy payment vehicle year made [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of Business [Source Data Type: varchar]"),
  `HomeOwner_flag` BOOL OPTIONS(description = "Ownership check of home [Source Data Type: varchar]"),
  `CIFState` STRING OPTIONS(description = "Customer address state as at application [Source Data Type: varchar]"),
  `Race` STRING OPTIONS(description = "Customer race as at application [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Customer gender as at application [Source Data Type: varchar]"),
  `Marital` STRING OPTIONS(description = "Customer marital status as at application [Source Data Type: varchar]"),
  `National` STRING OPTIONS(description = "Customer nationality as at application [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Customer occupation as at application [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Customer academic qualification as at application [Source Data Type: varchar]"),
  `B_AdvInstallPymt` FLOAT64 OPTIONS(description = "Easy payment advanced install payment [Source Data Type: numeric]"),
  `AdvInstallPymtBand` STRING OPTIONS(description = "Easy payment advanced installment payment band [Source Data Type: nvarchar]"),
  `JudgeWeek_FG` INT64 OPTIONS(description = "Judge week flag [Source Data Type: varchar]"),
  `Biometric_FG` BOOL OPTIONS(description = "Biometric indicator [Source Data Type: varchar]"),
  `E-KYC` STRING OPTIONS(description = "E-KYC pass/fail [Source Data Type: varchar]"),
  `OTP` STRING OPTIONS(description = "OTP pass/fail [Source Data Type: varchar]"),
  `APPLY_FIN_AMT` FLOAT64 OPTIONS(description = "Easy payment applied financing amount [Source Data Type: numeric]"),
  `PreAssessment_Flag` BOOL OPTIONS(description = "Some customers qualify for pre-assessment [Source Data Type: varchar]")
) OPTIONS(description = "The current (daily full refreshed) application status (Approved or Rejected) for EP products. (Governed via Mock Metadata.xlsx | Sheet: T1 - Fact_EP_Judge)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Sales` (
  `TX_DT` FLOAT64 OPTIONS(description = "Sales Date"),
  `CIF_No` STRING OPTIONS(description = "Unique customer ID"),
  `TransactionCountry` STRING OPTIONS(description = "Ringgit Malaysia"),
  `TransactionCountryHigherLevel` STRING OPTIONS(description = "Local or Oversea"),
  `PriviledgeMerchantsGrp` STRING OPTIONS(description = "Member merchant"),
  `LDESC` STRING OPTIONS(description = "Spend Location"),
  `Sales_Type` STRING OPTIONS(description = "Cash Purchase or Cash Advance"),
  `Amount` FLOAT64 OPTIONS(description = "Transaction Amount"),
  `TransCount` FLOAT64 OPTIONS(description = "Transaction Count")
) OPTIONS(description = "The confirmed sales log of the EP product. (Governed via Mock Metadata.xlsx | Sheet: T2 - Fact_EP_Sales)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Collection` (
  `TX_DT` INT64 OPTIONS(description = "Reporting date [Source Data Type: decimal]"),
  `First_INST_DT` INT64 OPTIONS(description = "First intallment date [Source Data Type: decimal]"),
  `Agree_No` INT64 OPTIONS(description = "Loan agreement ID [Source Data Type: varchar]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `Current_Time_Payment` INT64 OPTIONS(description = "current installment period [Source Data Type: varchar]"),
  `Del_Sts` INT64 OPTIONS(description = "Lock Deliquency status [Source Data Type: varchar]"),
  `Collection_Branch` INT64 OPTIONS(description = "Branch ID [Source Data Type: varchar]"),
  `Score_Value` INT64 OPTIONS(description = "Collection Score Point [Source Data Type: decimal]"),
  `Score_Grade` STRING OPTIONS(description = "Collection Score Grade [Source Data Type: varchar]"),
  `Sub_Code` STRING OPTIONS(description = "AKPK checker [Source Data Type: varchar]"),
  `Pay_in_Full` INT64 OPTIONS(description = "Account Status [Source Data Type: varchar]"),
  `Classification_Code` STRING OPTIONS(description = "Life Deliquency status [Source Data Type: varchar]"),
  `FinPlus_Code` STRING OPTIONS(description = "FinPlus (e-credit evaluation) tier [Source Data Type: varchar]"),
  `MDD` INT64 OPTIONS(description = "EP Multi Due Date [Source Data Type: varchar]"),
  `LoanTyp_ID` INT64 OPTIONS(description = "Loan product type indicator [Source Data Type: varchar]"),
  `Billing_OSP` FLOAT64 OPTIONS(description = "Principal Billing amount [Source Data Type: decimal]"),
  `Billing_Count` INT64 OPTIONS(description = "Principal Billing count [Source Data Type: decimal]"),
  `Collection_OSP` FLOAT64 OPTIONS(description = "Principal Collection amount [Source Data Type: decimal]"),
  `Collection_Count` INT64 OPTIONS(description = "Principal Collection count [Source Data Type: decimal]"),
  `Unpaid_OSP` FLOAT64 OPTIONS(description = "Principal Unpaid amount [Source Data Type: decimal]"),
  `Unpaid_Count` INT64 OPTIONS(description = "Principal Unpaid count [Source Data Type: decimal]")
) OPTIONS(description = "The collection status snapshot for EP products as at closing period (Governed via Mock Metadata.xlsx | Sheet: T3 - Fact_EP_Collection)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Judge` (
  `Rcd_DT` DATE OPTIONS(description = "Data extraction date [Source Data Type: date]"),
  `Account_No` INT64 OPTIONS(description = "Card account number [Source Data Type: varchar]"),
  `Appl_ID` INT64 OPTIONS(description = "Credit card application ID [Source Data Type: varchar]"),
  `ApplSts_ID` STRING OPTIONS(description = "Credit card application status ID [Source Data Type: varchar]"),
  `CIF_ID` INT64 OPTIONS(description = "Credit card customer ID [Source Data Type: varchar]"),
  `CardTyp_ID` STRING OPTIONS(description = "Credit card type [Source Data Type: varchar]"),
  `CardBrand_ID` STRING OPTIONS(description = "Credit card brand [Source Data Type: varchar]"),
  `CardApplTyp_ID` STRING OPTIONS(description = "Principal/supplementary [Source Data Type: varchar]"),
  `ApplChnnl_ID` STRING OPTIONS(description = "Credit card application channel [Source Data Type: varchar]"),
  `Reject_ID` INT64 OPTIONS(description = "Credit card rejection reason ID [Source Data Type: varchar]"),
  `Decline_ID` INT64 OPTIONS(description = "Rejection reason [Source Data Type: varchar]"),
  `Agent_ID` INT64 OPTIONS(description = "Merchant ID [Source Data Type: varchar]"),
  `ScoreDecision_ID` STRING OPTIONS(description = "Credit card score decision category [Source Data Type: varchar]"),
  `ScoreRank_ID` STRING OPTIONS(description = "Credit card score rank [Source Data Type: varchar]"),
  `SysRcmmd_ID` STRING OPTIONS(description = "System recommended decision [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Gender [Source Data Type: varchar]"),
  `Age` INT64 OPTIONS(description = "Age [Source Data Type: int]"),
  `Race` STRING OPTIONS(description = "Race [Source Data Type: varchar]"),
  `Nationality` STRING OPTIONS(description = "Nationality short code [Source Data Type: varchar]"),
  `MaritalSts` STRING OPTIONS(description = "Marital Status [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Highest academic qualification [Source Data Type: varchar]"),
  `YrStay` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: int]"),
  `HomeOwn` STRING OPTIONS(description = "Type of home ownership [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Occupation [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of business of customers employer [Source Data Type: varchar]"),
  `YrJob` INT64 OPTIONS(description = "Year in job [Source Data Type: int]"),
  `NetIncome` INT64 OPTIONS(description = "Net income [Source Data Type: int]"),
  `GrossIncome` INT64 OPTIONS(description = "Gross income [Source Data Type: int]"),
  `AnnualIncome` INT64 OPTIONS(description = "Annual income [Source Data Type: int]"),
  `AnnualIncomeTotLmt` INT64 OPTIONS(description = "Annual income total limit [Source Data Type: int]"),
  `CurrRepay` INT64 OPTIONS(description = "Current repayment amount [Source Data Type: int]"),
  `NewRepay` INT64 OPTIONS(description = "New repayment amount [Source Data Type: int]"),
  `RcmmdIntrst` FLOAT64 OPTIONS(description = "Recommended interest rate [Source Data Type: decimal]"),
  `NDI` INT64 OPTIONS(description = "Net Disposable Income [Source Data Type: int]"),
  `CurrDSR` INT64 OPTIONS(description = "Current DSR [Source Data Type: decimal]"),
  `NewDSR` INT64 OPTIONS(description = "New DSR [Source Data Type: decimal]"),
  `PaySlipTyp_ID` INT64 OPTIONS(description = "Payslip type [Source Data Type: varchar]"),
  `CardActivate_FG` BOOL OPTIONS(description = "Card activated [Source Data Type: varchar]"),
  `EmergencyCont_FG` BOOL OPTIONS(description = "Emergency contact provided [Source Data Type: varchar]"),
  `CardActivate_DT` INT64 OPTIONS(description = "Card activation date [Source Data Type: decimal]"),
  `Judge_DT` INT64 OPTIONS(description = "Decision date [Source Data Type: decimal]"),
  `Appl_DT` INT64 OPTIONS(description = "Application date [Source Data Type: decimal]"),
  `B_Limit` FLOAT64 OPTIONS(description = "Total limit [Source Data Type: decimal]"),
  `B_CrLimit` INT64 OPTIONS(description = "Credit limit [Source Data Type: decimal]"),
  `B_CashAdvLimit` FLOAT64 OPTIONS(description = "Cash advance limit [Source Data Type: decimal]"),
  `B_NonBankCommitment` FLOAT64 OPTIONS(description = "Non bank commitment [Source Data Type: decimal]"),
  `CIFState_ID` STRING OPTIONS(description = "State [Source Data Type: varchar]"),
  `Biometric_FG` STRING OPTIONS(description = "Biometric indicator [Source Data Type: varchar]"),
  `FinalDecline_ID` INT64 OPTIONS(description = "Decline ID [Source Data Type: varchar]"),
  `Final_Score` INT64 OPTIONS(description = "Credit card CTOS score [Source Data Type: decimal]"),
  `Final_Score_Type` INT64 OPTIONS(description = "Credit card CTOS score [Source Data Type: decimal]"),
  `Final_ScoreDesc` STRING OPTIONS(description = "Credit card CTOS score description [Source Data Type: varchar]"),
  `ApplyCardBiz_ID` STRING OPTIONS(description = "Applied card ID [Source Data Type: varchar]"),
  `ProposedCardBiz_ID` STRING OPTIONS(description = "Proposed card ID [Source Data Type: varchar]")
) OPTIONS(description = "The current (daily full refreshed) application status (Approved or Rejected) for credit cards. (Governed via Mock Metadata.xlsx | Sheet: T4 - Fact_CC_Judge)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Sales` (
  `TX_DT` INT64 OPTIONS(description = "Sales Date [Source Data Type: decimal]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `TransactionCountry` STRING OPTIONS(description = "Ringgit Malaysia [Source Data Type: varchar]"),
  `TransactionCountryHigherLevel` STRING OPTIONS(description = "Local or Oversea [Source Data Type: varchar]"),
  `PriviledgeMerchantsGrp` STRING OPTIONS(description = "Member merchant [Source Data Type: varchar]"),
  `LDESC` STRING OPTIONS(description = "Spend Location [Source Data Type: varchar]"),
  `Sales_Type` STRING OPTIONS(description = "Cash Purchase or Cash Advance [Source Data Type: varchar]"),
  `Amount` FLOAT64 OPTIONS(description = "Transaction Amount [Source Data Type: decimal]"),
  `TransCount` INT64 OPTIONS(description = "Transaction Count [Source Data Type: decimal]")
) OPTIONS(description = "The actual spending & cash advance log on credit card (Governed via Mock Metadata.xlsx | Sheet: T5 - Fact_CC_Sales)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Collection` (
  `TX_DT` INT64 OPTIONS(description = "Reporting date [Source Data Type: decimal]"),
  `Account_No` INT64 OPTIONS(description = "CC account ID [Source Data Type: varchar]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `DC_Sts` INT64 OPTIONS(description = "Lock deliquency status [Source Data Type: varchar]"),
  `Application_Branch` INT64 OPTIONS(description = "Branch ID [Source Data Type: varchar]"),
  `Score_Value` INT64 OPTIONS(description = "Collection Score point [Source Data Type: decimal]"),
  `Score_Grade` STRING OPTIONS(description = "Collection Score grade [Source Data Type: varchar]"),
  `MDD` INT64 OPTIONS(description = "CC Due Date [Source Data Type: varchar]"),
  `FinPlus_Code` STRING OPTIONS(description = "FinPlus (e-credit evaluation) tier [Source Data Type: varchar]"),
  `Billing_OSP` FLOAT64 OPTIONS(description = "Billing amount [Source Data Type: decimal]"),
  `Billing_Count` INT64 OPTIONS(description = "Billing count [Source Data Type: decimal]"),
  `Collection_OSP` FLOAT64 OPTIONS(description = "Collection amount [Source Data Type: decimal]"),
  `Collection_Count` INT64 OPTIONS(description = "Collection count [Source Data Type: decimal]"),
  `Unpaid_OSP` FLOAT64 OPTIONS(description = "Unpaid amount [Source Data Type: decimal]"),
  `Unpaid_Count` INT64 OPTIONS(description = "Unpaid count [Source Data Type: decimal]"),
  `Maintain_OSP` FLOAT64 OPTIONS(description = "Maintain amount [Source Data Type: decimal]"),
  `Maintain_Count` INT64 OPTIONS(description = "Maintain count [Source Data Type: decimal]")
) OPTIONS(description = "The billing and collection status snapshot for credit cards as at closing period (Governed via Mock Metadata.xlsx | Sheet: T6 - Fact_CC_Collection)");



> **🔍 How to Verify Step 4 on GCP Console UI (BigQuery Studio Explorer & Schema Tab)**
> 1. In the left **BigQuery Studio Explorer** pane (right beside this notebook!), expand **`${PROJECT_ID}` $\rightarrow$ `acsm_bronze`**.
> 2. Click on **`Fact_EP_Judge`** $\rightarrow$ select the **Schema** tab to verify all **60 columns** have their business **Description** populated from `Mock Metadata.xlsx`, and check the **Details** tab to confirm **Data location** is `asia-southeast1` and **Number of rows** is currently `0`.
---
## Step 5: Run Serverless `LOAD DATA OVERWRITE` Statement (**$0 Load Cost / `0 B Billed`**)
> **💡 Zero Compute Cost (`$0.00` / `0 Bytes Billed`)**: Batch loading data into BigQuery from Cloud Storage via the `LOAD DATA` SQL statement is **100% FREE (`$0.00`)** using BigQuery's shared batch slot pool ([BigQuery Pricing](https://cloud.google.com/bigquery/pricing#loading_data)). Because both the bucket and dataset are in **Singapore (`asia-southeast1`)**, there is **$0 network egress cost** and **100% of the 185 column descriptions across the 6 Native Fact tables** from Step 4 are preserved automatically.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- DEMO FLOW 2 (STEP 2 OF 2): Serverless SQL `LOAD DATA OVERWRITE` from GCS
-- Dynamically resolves `@@project_id` (`gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/`)
-- into the 6 pre-created BigQuery Native Fact tables in `acsm_bronze` (1,233,284 rows) while preserving all 185 column descriptions.
-- Zero compute provisioning required | $0 BigQuery batch load cost (0 B billed).
-- =============================================================================

DECLARE bucket_uri STRING DEFAULT CONCAT('gs://acsm-workshop-landing-', @@project_id, '/full_compressed');

-- 1. Load Fact_EP_Judge (140,000 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Judge`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_EP_Judge.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 2. Load Fact_EP_Sales (119,859 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Sales`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_EP_Sales.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 3. Load Fact_EP_Collection (80,000 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Collection`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_EP_Collection.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 4. Load Fact_CC_Judge (227,500 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Judge`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_CC_Judge.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 5. Load Fact_CC_Sales (535,925 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Sales`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_CC_Sales.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 6. Load Fact_CC_Collection (130,000 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Collection`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_CC_Collection.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);


---
## Step 6: Pure SQL Verification (`INFORMATION_SCHEMA` Audits — Zero Python)

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 1: Audit Row Counts, Table Descriptions & Described Columns (with Grand Total)
WITH table_stats AS (
  SELECT
    t.table_name,
    s.row_count,
    COUNT(c.column_name) AS total_columns,
    COUNTIF(c.description IS NOT NULL AND c.description != "") AS described_columns,
    REGEXP_REPLACE(COALESCE(opt.option_value, ""), r"^\"|\"$", "") AS table_description
  FROM `acsm_bronze.INFORMATION_SCHEMA.TABLES` t
  JOIN `acsm_bronze.__TABLES__` s
    ON t.table_name = s.table_id
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.TABLE_OPTIONS` opt
    ON t.table_name = opt.table_name AND opt.option_name = "description"
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS` c
    ON t.table_name = c.table_name
  WHERE t.table_name IN ("Fact_EP_Judge", "Fact_EP_Sales", "Fact_EP_Collection", "Fact_CC_Judge", "Fact_CC_Sales", "Fact_CC_Collection", "m3CIF", "dimProduct")
  GROUP BY 1, 2, 5
)
SELECT
  table_name,
  row_count,
  total_columns,
  described_columns,
  ROUND(SAFE_DIVIDE(described_columns, total_columns) * 100, 1) AS coverage_pct,
  table_description
FROM table_stats
UNION ALL
SELECT
  "TOTAL (ALL 8 ACSM TABLES)" AS table_name,
  SUM(row_count) AS row_count,
  SUM(total_columns) AS total_columns,
  SUM(described_columns) AS described_columns,
  ROUND(SAFE_DIVIDE(SUM(described_columns), SUM(total_columns)) * 100, 1) AS coverage_pct,
  "100% Serverless Load Complete | 0 Bytes Billed ($0.00)" AS table_description
FROM table_stats
ORDER BY CASE WHEN STARTS_WITH(table_name, "TOTAL") THEN 2 ELSE 1 END, table_name;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 2: Prove $0 Load Cost (0 Bytes Billed) via BigQuery INFORMATION_SCHEMA.JOBS
SELECT
  job_id,
  statement_type,
  destination_table.table_id AS loaded_table,
  state,
  COALESCE(total_bytes_billed, 0) AS bytes_billed,
  "$0.00 (Free Shared Batch Pool)" AS ingestion_compute_cost,
  TIMESTAMP_DIFF(end_time, start_time, SECOND) AS duration_seconds
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE statement_type = "LOAD_DATA"
  AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
ORDER BY creation_time DESC
LIMIT 8;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 3: Inspect Governed Column Descriptions from INFORMATION_SCHEMA
SELECT
  table_name,
  column_name,
  data_type,
  description
FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
WHERE table_name IN ("Fact_EP_Judge", "Fact_EP_Sales", "Fact_EP_Collection", "Fact_CC_Judge", "Fact_CC_Sales", "Fact_CC_Collection", "m3CIF", "dimProduct")
ORDER BY table_name, column_name
LIMIT 25;


> **🔍 How to Verify Step 5 on GCP Console UI (4 Visual Checks in BigQuery Studio)**
> 1. **Verify `$0` Load Cost (`0 B Billed`)**: In the cell output above (or in BigQuery **Job history**), point out **`Total Bytes Billed: 0 B ($0.00 FREE Serverless Batch Load)`** in **`asia-southeast1`**.
> 2. **Verify Row Counts (`Details` Tab)**: In the left Explorer tree, click **`Fact_CC_Sales`** ($535,925$ rows) or **`Fact_EP_Judge`** ($140,000$ rows) $\rightarrow$ **Details** tab.
> 3. **Verify Loaded Records (`Preview` Tab — also $0 Cost)**: Click the **Preview** tab on any table to browse the records at zero query cost (`0 B billed`).
> 4. **Verify Preserved Column Descriptions (`Schema` Tab)**: Click the **Schema** tab to confirm all **185 column descriptions across the 6 Native Fact tables** remained intact after `LOAD DATA OVERWRITE`.

---
## Step 7: Storage Engine 2 — Convert `acsm_bronze.m3CIF` (`100,000` rows) into a GCP Lakehouse Open-Source Apache Iceberg Table (`C1.1.1.3`, `C1.1.1.4`, `C1.1.1.19`)

Now that our Fact tables are loaded into BigQuery Native Storage, we convert **`acsm_bronze.m3CIF` (`100,000` rows, Customer CIF Master)** into a **GCP BigLake Managed Apache Iceberg Table (`table_format = 'ICEBERG'`, `file_format = 'PARQUET'`)** stored on Google Cloud Storage (`gs://acsm-workshop-landing-${PROJECT_ID}/iceberg/m3CIF`).

### What the Cell Below Executes:
1. Auto-creates the BigLake Cloud Resource Connection (`acsm-biglake-iceberg-conn`) in `asia-southeast1` and grants `roles/storage.objectAdmin` to its Google-managed Service Account.
2. Replaces `acsm_bronze.m3CIF` with a **GCP Lakehouse Apache Iceberg table** (`WITH CONNECTION ... OPTIONS (file_format = 'PARQUET', table_format = 'ICEBERG', storage_uri = 'gs://...')`) preserving all **28 Dataplex business glossary column descriptions** and loading all **`100,000` customer records**.
3. Runs `EXPORT TABLE METADATA FROM acsm_bronze.m3CIF` and lists the open `.parquet` data files and Iceberg `metadata/v*.metadata.json` manifest files in your GCS bucket!
4. Demonstrates zero-rewrite **Schema Evolution (`ALTER TABLE acsm_bronze.m3CIF ADD COLUMN`)** and **Time Travel (`FOR SYSTEM_TIME AS OF`)**.


In [ ]:
import json
import time

BIGLAKE_CONN_ID = 'acsm-biglake-iceberg-conn'
M3CIF_ICEBERG_URI = f'gs://{BUCKET_NAME}/iceberg/m3CIF'

# 1. Auto-create BigLake Cloud Resource Connection in asia-southeast1
print(f'1. Ensuring BigLake Connection `{BIGLAKE_CONN_ID}` exists in {LOCATION}...')
conn_list = subprocess.run(
    ['bq', 'ls', '--connection', f'--project_id={PROJECT_ID}', f'--location={LOCATION}', '--format=json'],
    capture_output=True, text=True
)
if BIGLAKE_CONN_ID not in conn_list.stdout:
    subprocess.run([
        'bq', 'mk', '--connection', '--connection_type=CLOUD_RESOURCE',
        f'--project_id={PROJECT_ID}', f'--location={LOCATION}', BIGLAKE_CONN_ID
    ], check=True)
    print(f'   Created BigLake connection: {BIGLAKE_CONN_ID}')
else:
    print(f'   BigLake connection `{BIGLAKE_CONN_ID}` already exists.')

# Grant Storage Object Admin to BigLake Connection Service Account
conn_info_raw = subprocess.check_output([
    'bq', 'show', '--connection', '--format=json',
    f'{PROJECT_ID}.{LOCATION}.{BIGLAKE_CONN_ID}'
], text=True)
conn_sa = json.loads(conn_info_raw).get('cloudResource', {}).get('serviceAccountId', '')
print(f'2. Granting roles/storage.objectAdmin on gs://{BUCKET_NAME} to BigLake SA: {conn_sa}')
if conn_sa:
    subprocess.run([
        'gcloud', 'storage', 'buckets', 'add-iam-policy-binding', f'gs://{BUCKET_NAME}',
        f'--member=serviceAccount:{conn_sa}',
        '--role=roles/storage.objectAdmin',
        '--quiet'
    ], check=False)
    time.sleep(5)

# 2. Stage current m3CIF rows, re-create `acsm_bronze.m3CIF` as a GCP Lakehouse Apache Iceberg Table, and populate 100,000 rows
m3cif_iceberg_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.m3CIF` (
  CIF_ID INT64 OPTIONS(description='Unique Customer Information File identifier across all AEON products'),
  Rcd_DT STRING OPTIONS(description='Record creation or latest profile refresh date (YYYY-MM-DD)'),
  CIF_NM STRING OPTIONS(description='Full legal name of the customer (PII - subject to PDPA masking)'),
  MaritalSts STRING OPTIONS(description='Marital status of the customer (Single, Married, Divorced, Widowed)'),
  Gender STRING OPTIONS(description='Customer gender (M / F)'),
  Citizen STRING OPTIONS(description='Citizenship status (Malaysian, Permanent Resident, Foreigner)'),
  State STRING OPTIONS(description='Malaysian state of residence (e.g. Selangor, Johor, Kuala Lumpur, Penang)'),
  Region STRING OPTIONS(description='Geographical region grouping (Central, Northern, Southern, East Coast, East Malaysia)'),
  Race STRING OPTIONS(description='Demographic race/ethnicity classification for BNM regulatory reporting'),
  Buss_Nature STRING OPTIONS(description='Industry or business sector of the customer employer'),
  HomeOwnership STRING OPTIONS(description='Residential ownership status (Owned, Rented, Family, Company Provided)'),
  HomePostcode INT64 OPTIONS(description='5-digit Malaysian postal code of residential address'),
  _HomeAddr1 STRING OPTIONS(description='Primary street address line 1 of the customer (PII - subject to PDPA masking)'),
  EmpSts INT64 OPTIONS(description='Employment status code (1=Permanent, 2=Contract, 3=Self-Employed, 4=Unemployed)'),
  EmployerNM STRING OPTIONS(description='Registered name of the customer current employer'),
  Occupation STRING OPTIONS(description='Job title or occupational category of the customer'),
  Academic STRING OPTIONS(description='Highest academic qualification attained'),
  _SelfEmp_FG STRING OPTIONS(description='Flag indicating if customer is self-employed (Y/N)'),
  FELDA_FG STRING OPTIONS(description='Flag indicating FELDA settler/program participant status (Y/N)'),
  JointIncome_FG STRING OPTIONS(description='Flag indicating whether application includes joint/spousal income (Y/N)'),
  RecvPromo_FG STRING OPTIONS(description='PDPA marketing consent flag to receive promotional offers (Y/N)'),
  N_YrStay FLOAT64 OPTIONS(description='Number of years residing at current residential address'),
  N_Age INT64 OPTIONS(description='Customer age in years'),
  N_Depend INT64 OPTIONS(description='Number of financial dependents declared by customer'),
  N_YrJob FLOAT64 OPTIONS(description='Number of years in current employment or business'),
  B_NetIncome FLOAT64 OPTIONS(description='Verified monthly net income in MYR'),
  B_GrossIncome FLOAT64 OPTIONS(description='Declared monthly gross income in MYR'),
  B_AnnualIncome FLOAT64 OPTIONS(description='Computed total annual income in MYR')
)
CLUSTER BY State, Citizen
WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{BIGLAKE_CONN_ID}`
OPTIONS (
  file_format = 'PARQUET',
  table_format = 'ICEBERG',
  storage_uri = '{M3CIF_ICEBERG_URI}',
  description = 'ACSM Customer Master (m3CIF — 100,000 rows) stored as a GCP Lakehouse Open-Source Apache Iceberg Table (Parquet + Iceberg metadata on GCS)'
);
"""
client.query(m3cif_iceberg_sql).result()

# Populate the 100,000 customer rows into the Apache Iceberg table
load_m3cif_iceberg = f"""
LOAD DATA OVERWRITE `{PROJECT_ID}.{DATASET_ID}.m3CIF`
FROM FILES (
  format = 'CSV',
  uris = ['gs://{BUCKET_NAME}/full_compressed/m3CIF.csv.gz'],
  skip_leading_rows = 1
);
"""
client.query(load_m3cif_iceberg).result()
client.query(f'EXPORT TABLE METADATA FROM `{PROJECT_ID}.{DATASET_ID}.m3CIF`').result()
print('3. Converted `acsm_bronze.m3CIF` (100,000 rows) into a GCP Lakehouse Apache Iceberg Table and exported metadata.json to GCS!')

print('\n--- Open Apache Parquet & Iceberg Metadata Files in gs://.../iceberg/m3CIF ---')
gcs_iceberg_files = subprocess.run(['gcloud', 'storage', 'ls', '--recursive', f'{M3CIF_ICEBERG_URI}/'], capture_output=True, text=True)
for line in gcs_iceberg_files.stdout.strip().splitlines()[:12]:
    print('  ', line)


---
## Step 8: Storage Engine 3 — Federate Cross-Cloud AWS Glue Iceberg Table (`acsm_aws_bronze.dimProduct` — `65,000` rows) into BigQuery (`C1.1.1.1`, `C1.1.1.3`)

Finally, we connect BigQuery directly to the **AWS Glue Data Catalog (`AWS Account: <AWS_ACCOUNT_ID>`, `IAM Role: acsm-gcp-trust-role`, `Policy: acsm-federated-only-policy`, `ap-southeast-1` AWS Singapore)** where **`acsm_aws_bronze.dimProduct` (`65,000` rows, Credit Card Product Master)** is stored as an **AWS S3 / AWS Glue Apache Iceberg table (`s3://...`)**.

### The 3-Step AWS Glue Iceberg Federation Flow:
1. **Step 8.1 — Create the BigLake Federated Catalog (`acsm_aws_federated_catalog`)**:
   ```bash
   export AWS_ACCOUNT_ID=<AWS_ACCOUNT_ID>
   export AWS_ROLE_NAME=acsm-gcp-trust-role

   gcloud alpha biglake iceberg catalogs create acsm_aws_federated_catalog \
       --project="<PROJECT_ID>" \
       --primary-location="asia-southeast1" \
       --catalog-type="federated" \
       --federated-catalog-type="glue" \
       --refresh-interval="300s" \
       --glue-warehouse="$AWS_ACCOUNT_ID" \
       --glue-aws-region="ap-southeast-1" \
       --glue-aws-role-arn="arn:aws:iam::$AWS_ACCOUNT_ID:role/$AWS_ROLE_NAME"
   ```
2. **Step 8.2 — Get Your `biglake-service-account-id` (`blirc-...`) & Paste It in the Workshop Google Sheet**:
   ```bash
   gcloud alpha biglake iceberg catalogs describe acsm_aws_federated_catalog \
       --project="<PROJECT_ID>" \
       --primary-location="asia-southeast1" \
       --format="value(biglake-service-account-id)"
   ```
   👉 Paste your returned Service Account ID into the **[Workshop AWS Trust Policy Google Sheet](https://docs.google.com/spreadsheets/d/1oWyKOTAXlzLTMKrNPw3xa9l4XwGCg84aF6aP_XtYQcc/edit?resourcekey=0-M6VZHHykIa1ul5ZL0y7U1g&gid=0#gid=0)** so the instructor can add your Service Account ID to the AWS IAM Role (`arn:aws:iam::<AWS_ACCOUNT_ID>:role/acsm-gcp-trust-role`) Web Identity Trust Policy (`"accounts.google.com:sub"`).
3. **Step 8.3 — Verify `acsm_aws_bronze.dimProduct` in the BigQuery Studio UI & Query Live**:
   - In **BigQuery Studio Explorer**, expand **`acsm_aws_federated_catalog` → `acsm_aws_bronze` → `dimProduct`**.
   - Notice that **ONLY `acsm_aws_bronze.dimProduct`** is visible in the BigQuery UI because `acsm-gcp-trust-role` (`acsm-federated-only-policy`) is scoped strictly to `arn:aws:glue:ap-southeast-1:<AWS_ACCOUNT_ID>:table/acsm_aws_bronze/dimProduct`!
   - Click the **Details** tab to verify **Table Format: `ICEBERG`** and **Storage URI: `s3://...` (`ap-southeast-1` Singapore)**!


In [ ]:
# Default AWS Glue Federation Parameters (AWS Singapore Region: ap-southeast-1)
AWS_ACCOUNT_ID = os.environ.get('AWS_ACCOUNT_ID', '<AWS_ACCOUNT_ID>')
AWS_ROLE_NAME = os.environ.get('AWS_ROLE_NAME', 'acsm-gcp-trust-role')
AWS_REGION = os.environ.get('AWS_REGION', 'ap-southeast-1')
AWS_ROLE_ARN = f'arn:aws:iam::{AWS_ACCOUNT_ID}:role/{AWS_ROLE_NAME}'
FEDERATED_CATALOG_NAME = os.environ.get('FEDERATED_CATALOG_NAME', 'acsm_aws_federated_catalog')
FEDERATED_PRIMARY_LOCATION = os.environ.get('FEDERATED_PRIMARY_LOCATION', 'asia-southeast1')
AWS_GLUE_DATABASE = os.environ.get('AWS_GLUE_DATABASE', 'acsm_aws_bronze')
AWS_ICEBERG_TABLE = os.environ.get('AWS_ICEBERG_TABLE', 'dimProduct')
SA_TRACKER_SHEET_URL = 'https://docs.google.com/spreadsheets/d/1oWyKOTAXlzLTMKrNPw3xa9l4XwGCg84aF6aP_XtYQcc/edit?resourcekey=0-M6VZHHykIa1ul5ZL0y7U1g&gid=0#gid=0'

subprocess.run(['gcloud', 'services', 'enable', 'biglake.googleapis.com', f'--project={PROJECT_ID}', '--quiet'], check=False)

create_catalog_cmd = [
    'gcloud', 'alpha', 'biglake', 'iceberg', 'catalogs', 'create', FEDERATED_CATALOG_NAME,
    f'--project={PROJECT_ID}',
    f'--primary-location={FEDERATED_PRIMARY_LOCATION}',
    '--catalog-type=federated',
    '--federated-catalog-type=glue',
    '--refresh-interval=300s',
    f'--glue-warehouse={AWS_ACCOUNT_ID}',
    f'--glue-aws-region={AWS_REGION}',
    f'--glue-aws-role-arn={AWS_ROLE_ARN}'
]
print('1. Creating BigLake Federated Catalog for AWS Glue:\n  ' + ' \\\n    '.join(create_catalog_cmd) + '\n')
res_create = subprocess.run(create_catalog_cmd, capture_output=True, text=True)
if res_create.returncode == 0:
    print('✅ Created BigLake Federated Catalog:', FEDERATED_CATALOG_NAME)
elif 'ALREADY_EXISTS' in res_create.stderr or 'already exists' in res_create.stderr.lower():
    print(f'ℹ️ Federated Catalog `{FEDERATED_CATALOG_NAME}` already exists in `{PROJECT_ID}`.')
else:
    print('ℹ️ Output:', res_create.stderr.strip() or res_create.stdout.strip())

# Retrieve the participant's BigLake Service Account ID (`biglake-service-account-id`)
describe_cmd = [
    'gcloud', 'alpha', 'biglake', 'iceberg', 'catalogs', 'describe', FEDERATED_CATALOG_NAME,
    f'--project={PROJECT_ID}',
    f'--primary-location={FEDERATED_PRIMARY_LOCATION}',
    '--format=value(biglake-service-account-id)'
]
res_desc = subprocess.run(describe_cmd, capture_output=True, text=True)
biglake_sa_id = res_desc.stdout.strip()
if not biglake_sa_id:
    res_desc_alt = subprocess.run([
        'gcloud', 'alpha', 'biglake', 'iceberg', 'catalogs', 'describe', FEDERATED_CATALOG_NAME,
        f'--project={PROJECT_ID}',
        '--format=value(biglake-service-account-id)'
    ], capture_output=True, text=True)
    biglake_sa_id = res_desc_alt.stdout.strip()

print('\n********************************************************************************')
print(f'🔑 YOUR BIGLAKE SERVICE ACCOUNT ID : {biglake_sa_id or "<Run command in Cloud Shell if prompted>"}')
print(f'📋 PASTE YOUR SA ID IN THIS SHEET  : {SA_TRACKER_SHEET_URL}')
print(f'🛡️ AWS IAM ROLE TO TRUST YOUR SA   : {AWS_ROLE_ARN}')
print(f'🧊 AWS GLUE ICEBERG TABLE IN BQ UI : `{PROJECT_ID}.{FEDERATED_CATALOG_NAME}.{AWS_GLUE_DATABASE}.{AWS_ICEBERG_TABLE}`')
print('********************************************************************************')

print('\nOnce the instructor updates the AWS Trust Policy, run this SQL query in BigQuery:')
print(f"""SELECT *
FROM `{PROJECT_ID}.{FEDERATED_CATALOG_NAME}.{AWS_GLUE_DATABASE}.{AWS_ICEBERG_TABLE}`
LIMIT 20;""")
